In [29]:
import json
from pathlib import Path
import os
import numpy as np
import re
import random
from collections import defaultdict, Counter

In [5]:
def read_jsonl(path:Path):
    data = []
    with path.open("r") as f:
        for line in f:
            data.append(json.loads(line))

    return data

path_to_format_raw = Path("../results/LLM_as_judge/label_inference_task/inferred_format_labels_raw_descriptors.jsonl")
path_to_topic_raw = Path("../results/LLM_as_judge/label_inference_task/inferred_topic_labels_raw_descriptors.jsonl")
path_to_topic_harmonized = Path("../results/LLM_as_judge/label_inference_task/inferred_topic_labels_harmonized_descriptors.jsonl")
format_raw = read_jsonl(path_to_format_raw)
topic_raw = read_jsonl(path_to_topic_raw)
topic_harmonized = read_jsonl(path_to_topic_harmonized)

In [30]:
def parse_response(response: str):
    text = response.strip().lower()
    if "answer:" in text:
        _, suffix = text.split("answer:", 1)
        answer = suffix.strip(" *:!?.-\n\r\t")
    else:
        answer = ""
        
    return re.sub(r"[\[\]\"']", "", answer)

def normalize_label(label):
    label = label.strip().lower()
    if label == "about (personal)":
        return "about (pers.)"
    elif label == "science & technology":
        return "science & tech."
    elif label == "software development":
        return "software dev."
    else:
        return label

In [36]:
label_types = ["format", "topic"]
descriptor_types = ["raw", "harmonized"]

for label_type in label_types:
    for desc_type in descriptor_types:
        correct = 0
        false = 0
        labels = defaultdict(list)
        misclassifications = defaultdict(Counter)
        path_to_results = Path(f"../results/LLM_as_judge/label_inference_task/inferred_{label_type}_labels_{desc_type}_descriptors.jsonl")
        if path_to_results.exists():
            results = read_jsonl(path_to_results)
        else:
            continue        
        for doc in results:
            true_label = doc["example"]["true_label"].lower().strip()
            true_label = normalize_label(true_label)
            labels[true_label]
            predicted_labels = parse_response(doc["response"]).split(",")
            predicted_labels = [normalize_label(l) for l in predicted_labels]
            if true_label in predicted_labels:
                correct += 1
                labels[true_label].append(True)
            else:
                false += 1
                labels[true_label].append(False)
                misclassifications[true_label].update(predicted_labels)
            


        print(label_type)
        print(desc_type)
        print("Correct:", correct)
        print("False:", false)
        print("Acc:", correct / (correct+false))
        print("True classifications:")
        for k,v in labels.items():
            t = len([t for t in v if t])
            f = len([l for l in v if not l])
            print(f"{k}: {round(t / (len(v)),2)}")
        print("======")
        print("Misclassifications:")
        for k,v in misclassifications.items():
            print(k)
            print(v.most_common(3))
            print()
        print()

format
raw
Correct: 56827
False: 43173
Acc: 0.56827
True classifications:
nonfiction writing: 0.82
news article: 0.57
about (pers.): 0.56
legal notices: 0.82
product page: 0.61
tutorial: 0.78
personal blog: 0.78
news (org.): 0.32
structured data: 0.22
comment section: 0.19
documentation: 0.8
content listing: 0.17
spam / ads: 0.1
academic writing: 0.89
audio transcript: 0.13
about (org.): 0.66
listicle: 0.25
faq: 0.34
knowledge article: 0.88
creative writing: 0.85
customer support: 0.22
truncated: 0.01
user review: 0.77
q&a forum: 0.33
Misclassifications:
about (pers.)
[('nonfiction writing', 513), ('personal blog', 342), ('knowledge article', 307)]

news article
[('knowledge article', 3509), ('nonfiction writing', 3369), ('news (org.)', 1756)]

structured data
[('knowledge article', 844), ('nonfiction writing', 551), ('documentation', 275)]

comment section
[('personal blog', 1232), ('nonfiction writing', 1047), ('q&a forum', 912)]

documentation
[('knowledge article', 79), ('tutorial'